In [ ]:
"""
================================================================================
CRESTA MLE INTERVIEW — PRACTICAL CODING PROBLEM
================================================================================
DOMAIN:  RAG Knowledge Retrieval for Agent Assist
TIME:    60-90 minutes
LEVEL:   Mid-Senior MLE

SCENARIO
--------
You are an MLE on Cresta's Agent Assist team. During live calls, human agents
need instant answers from internal knowledge bases (KB) — product docs, policy
manuals, troubleshooting guides, etc.

The "Knowledge Agent" watches the live conversation and, when the customer asks
a question, retrieves the most relevant KB article and generates a grounded
answer for the agent to read aloud or adapt.

CONSTRAINTS:
  - Latency budget: < 800ms end-to-end (retrieval + generation)
  - The KB has ~50 articles, each 100-500 words
  - Answers MUST cite their source article (no hallucination)
  - If the KB doesn't contain the answer, say "not found" — don't guess

YOUR TASK (4 parts)
-------------------
Part 1: Build the knowledge base + chunking              [10 min]
Part 2: Implement retrieval (TF-IDF similarity search)   [20 min]
Part 3: Generate grounded answers                        [15 min]
Part 4: Evaluate retrieval + answer quality              [15 min]

NOTE: No external APIs needed. We use TF-IDF vectors as embeddings and
mock_llm_call() for generation. In production you'd swap these for
sentence-transformers + Claude/GPT, but the architecture is identical.
================================================================================
"""

import re
import math
import json
from typing import Dict, List, Tuple, Optional
from collections import Counter
from dataclasses import dataclass, field


# ─────────────────────────────────────────────────────────────────────────────
# KNOWLEDGE BASE — 12 support articles for a fictional telecom company
# (do NOT modify)
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class Article:
    id: str
    title: str
    category: str
    content: str


@dataclass
class Chunk:
    chunk_id: str
    article_id: str
    article_title: str
    text: str
    position: int  # 0-indexed chunk number within the article


KB_ARTICLES = [
    Article(
        id="KB001",
        title="How to Reset Your Modem",
        category="troubleshooting",
        content="""If your internet connection is slow or dropping, resetting your modem often resolves the issue. Follow these steps:

Step 1: Locate the reset button on the back of your modem. It is a small recessed button that requires a paperclip or pin to press.

Step 2: Press and hold the reset button for 10 seconds. The lights on the modem will flash and then turn off.

Step 3: Wait 60 seconds for the modem to fully reboot. All lights should return to solid green.

Step 4: Test your connection by opening a web browser. If the issue persists, try connecting directly with an ethernet cable to rule out WiFi issues.

If the problem continues after resetting, there may be a line issue. Contact technical support to schedule a technician visit. The typical wait time for a technician appointment is 24-48 hours.""",
    ),
    Article(
        id="KB002",
        title="Billing Cycle and Payment Due Dates",
        category="billing",
        content="""Your billing cycle runs from the 1st to the last day of each month. Statements are generated on the 1st and payment is due by the 15th of the same month.

Payment methods accepted: credit card, debit card, bank transfer (ACH), and check. AutoPay can be set up through your online account or by calling customer service.

Late payments incur a $9.99 late fee if not received within 5 days of the due date. After 30 days past due, service may be suspended. After 60 days, the account is sent to collections.

If you need a payment extension, you can request one through the app or by calling us. Extensions of up to 10 days are available once per quarter. The extension waives the late fee but does not extend the grace period for the following month.""",
    ),
    Article(
        id="KB003",
        title="Plan Comparison: Basic vs Plus vs Premium",
        category="plans",
        content="""We offer three tiers of service:

Basic Plan — $49/month: 100 Mbps download speed, 10 Mbps upload, 1 TB data cap, standard WiFi router included. Best for individuals and light usage.

Plus Plan — $69/month: 300 Mbps download, 30 Mbps upload, unlimited data, enhanced WiFi 6 router included. Includes free access to our streaming partner bundle. Best for families and remote workers.

Premium Plan — $99/month: 1 Gbps download, 100 Mbps upload, unlimited data, WiFi 6E mesh system (3 nodes) included. Includes premium streaming bundle plus 100 GB cloud storage. Priority technical support with guaranteed 4-hour response time. Best for power users and large households.

All plans include a 30-day money-back guarantee. Contract terms are month-to-month with no early termination fees. Promotional pricing may apply for the first 12 months.""",
    ),
    Article(
        id="KB004",
        title="How to Cancel Your Service",
        category="account",
        content="""To cancel your Meridian Wireless service, you can call customer service or visit a retail store. Cancellations cannot be processed through the app or website.

When cancelling, please have your account number and the account holder's name ready. If you are on an equipment installment plan, the remaining balance will be charged to your final bill.

Cancellation takes effect at the end of your current billing cycle. You will continue to have service until that date and will not receive a prorated refund for the remaining days.

If you are cancelling due to a service issue, please let us know — we may be able to offer a resolution or a discounted rate to keep your service. Our retention team can provide exclusive offers not available elsewhere.

After cancellation, you must return all leased equipment within 14 days to avoid equipment charges of up to $150 per device.""",
    ),
    Article(
        id="KB005",
        title="International Calling and Roaming",
        category="features",
        content="""International calling is available as an add-on for $15/month. This includes unlimited calls to 60+ countries including Canada, Mexico, UK, Germany, France, Japan, and Australia. Calls to countries not on the included list are billed at $0.25 per minute.

International roaming is separate from international calling. When traveling abroad, roaming charges apply unless you purchase a travel pass. Travel passes are available in 3 options:

Day Pass — $10/day: Unlimited calls, texts, and 500 MB data in 30+ countries.
Week Pass — $40/week: Unlimited calls, texts, and 3 GB data in 30+ countries.
Month Pass — $80/month: Unlimited calls, texts, and 10 GB data in 50+ countries.

To activate a travel pass, text TRAVEL to 611 before your departure. Passes auto-expire and do not auto-renew. Without a travel pass, roaming charges are $2.99/minute for calls and $10/MB for data.""",
    ),
    Article(
        id="KB006",
        title="Equipment Return Policy",
        category="account",
        content="""All leased equipment must be returned within 14 days of service cancellation or equipment upgrade. Equipment includes modems, routers, mesh nodes, and TV boxes.

Return options: Drop off at any Meridian retail store, or request a prepaid shipping label through your online account. Shipping labels are emailed within 24 hours.

Equipment must be returned in working condition with all original cables and power adapters. Damaged or missing equipment will incur charges:

Modem: $100 replacement fee
Router: $75 replacement fee
Mesh node (each): $50 replacement fee
TV box: $125 replacement fee

Equipment charges appear on your final bill. If equipment is returned after the 14-day window but within 30 days, a $25 late return fee applies per device. After 30 days, the equipment is considered unreturned and full replacement charges apply.""",
    ),
    Article(
        id="KB007",
        title="Troubleshooting Slow Internet Speeds",
        category="troubleshooting",
        content="""If you are experiencing slow internet speeds, try these steps before contacting support:

1. Run a speed test at speedtest.meridianwireless.com while connected via ethernet cable. This establishes your baseline speed without WiFi interference.

2. If ethernet speed is normal but WiFi is slow, the issue is likely WiFi-related. Move your router to a central location, away from walls, microwaves, and other electronics. Ensure your device supports your router's frequency band (2.4 GHz or 5 GHz).

3. Check how many devices are connected. Each connected device shares your bandwidth. On the Basic plan (100 Mbps), more than 5 active streaming devices will cause noticeable slowdowns.

4. Restart your modem and router by unplugging them for 30 seconds, then plugging the modem in first, waiting 60 seconds, and then plugging in the router.

5. Check for firmware updates in your router's admin panel (usually accessible at 192.168.1.1).

If speeds are still below 80% of your plan's advertised speed on a wired connection, contact technical support for a line diagnostic.""",
    ),
    Article(
        id="KB008",
        title="How to Set Up AutoPay",
        category="billing",
        content="""AutoPay automatically charges your payment method on your due date each month, ensuring you never miss a payment.

To set up AutoPay online: Log in to your account at my.meridianwireless.com, go to Billing > Payment Methods, add or select a payment method, and toggle AutoPay on.

To set up by phone: Call customer service and provide your payment details. The agent will enable AutoPay on your account.

AutoPay customers receive a $5/month discount on all plans. The discount appears on your next statement after enrollment.

You can cancel AutoPay at any time through your online account or by calling us. Cancellation takes effect on the next billing cycle. Note that cancelling AutoPay will remove the $5 monthly discount.

If an AutoPay charge fails (e.g., expired card, insufficient funds), you will receive an email and text notification. You have 3 days to update your payment method before the late fee policy applies.""",
    ),
    Article(
        id="KB009",
        title="Service Outage Information",
        category="troubleshooting",
        content="""To check if there is a service outage in your area, visit status.meridianwireless.com or text OUTAGE to 611. You can also check the Meridian app under Service Status.

During an outage, you will receive automatic updates via text and email if you are opted into notifications. Estimated restoration times are provided when available but may change.

If the outage lasts more than 4 hours, affected customers on the Premium plan receive a prorated service credit automatically. Customers on Basic and Plus plans can request a credit by calling customer service or submitting a request through the app within 7 days of the outage.

For planned maintenance, we send advance notice at least 48 hours before the maintenance window. Planned maintenance typically occurs between 2 AM and 6 AM local time and lasts 1-2 hours.

If your service is not restored after an outage is marked as resolved, restart your modem. If that doesn't work, contact support — your individual connection may need to be re-provisioned.""",
    ),
    Article(
        id="KB010",
        title="Moving to a New Address",
        category="account",
        content="""If you are moving, you can transfer your Meridian Wireless service to your new address. Service availability varies by location.

To initiate a transfer: Call customer service or visit a retail store at least 7 days before your move date. Provide your new address so we can verify service availability and schedule installation.

If service is available at your new address, a technician visit is typically scheduled within 3-5 business days of your move date. There is no charge for the transfer if you keep the same plan. If you upgrade during the transfer, standard pricing applies.

If service is NOT available at your new address, you can cancel without any equipment fees or penalties — this counts as a no-fault cancellation.

During the transition, there may be a gap in service of 1-3 days. You will not be charged for days without service. Your account number and login credentials remain the same after the move.""",
    ),
    Article(
        id="KB011",
        title="Parental Controls and Content Filtering",
        category="features",
        content="""Meridian Wireless offers free parental controls on all plans through the Meridian Safe app.

Features include: website category blocking (adult content, gambling, social media, gaming), per-device time limits, bedtime scheduling (automatically disable internet for selected devices), and weekly usage reports emailed to the account holder.

To set up: Download the Meridian Safe app, log in with your Meridian account credentials, and follow the setup wizard. You can manage up to 20 devices.

Content filtering levels: Light (blocks adult and malware sites only), Moderate (adds gambling, drugs, and violence), Strict (adds social media and gaming). Custom filtering is available by adding individual domains to a blocklist.

Parental controls apply at the network level and work on all devices connected to your home network, including guest devices. They do not apply when devices are on cellular data or other WiFi networks.

To temporarily disable controls for a specific device, use the Pause feature in the app (requires the account holder's PIN).""",
    ),
    Article(
        id="KB012",
        title="Referral Program and Loyalty Rewards",
        category="billing",
        content="""Earn rewards for referring friends and family to Meridian Wireless.

How it works: Share your unique referral code (found in your account settings or the app). When a new customer signs up using your code, both you and the new customer receive a $50 account credit after their first 30 days of active service.

There is no limit to the number of referrals. Top referrers are eligible for our annual Champions Program, which includes exclusive perks like early access to new products and a dedicated support line.

Loyalty rewards: After 12 months of continuous service, you automatically join our Loyalty tier, which includes: a free speed boost of 50 Mbps on your current plan, priority customer support, and waived fees for one equipment replacement per year.

After 24 months, you reach Gold loyalty tier: all Silver benefits plus a $10/month loyalty discount, free premium WiFi router upgrade, and annual service review with a dedicated account manager.""",
    ),
]


# ─────────────────────────────────────────────────────────────────────────────
# CUSTOMER QUERIES — what the agent hears during a live call
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class Query:
    id: str
    text: str                # what the customer said
    expected_article: str    # KB article ID that should be retrieved
    expected_answer_keywords: List[str]  # key facts the answer should contain


TEST_QUERIES = [
    Query("q1", "How do I reset my modem? The internet keeps dropping.",
          "KB001", ["reset button", "10 seconds", "60 seconds", "paperclip"]),

    Query("q2", "What's the difference between your Basic and Plus plans?",
          "KB003", ["$49", "$69", "100 Mbps", "300 Mbps", "unlimited data"]),

    Query("q3", "When is my payment due and what happens if I'm late?",
          "KB002", ["15th", "$9.99", "5 days", "30 days"]),

    Query("q4", "I'm going to Europe next month. What are my options for using my phone?",
          "KB005", ["travel pass", "Day Pass", "$10/day", "Week Pass", "roaming"]),

    Query("q5", "How do I return the equipment after cancelling?",
          "KB006", ["14 days", "retail store", "prepaid shipping label", "replacement fee"]),

    Query("q6", "My internet is really slow but only on WiFi, the ethernet is fine.",
          "KB007", ["WiFi", "central location", "2.4 GHz", "5 GHz", "devices"]),

    Query("q7", "I'm moving next month, can I take my service with me?",
          "KB010", ["7 days", "technician", "3-5 business days", "no-fault cancellation"]),

    Query("q8", "Is there a way to get a discount on my bill?",
          "KB008", ["AutoPay", "$5/month discount"]),

    Query("q9", "I want to block certain websites for my kids' devices.",
          "KB011", ["Meridian Safe", "parental controls", "per-device", "bedtime"]),

    Query("q10", "There's no internet at all — is there an outage?",
          "KB009", ["status.meridianwireless.com", "text OUTAGE to 611", "credit"]),

    # Harder: query that no article fully answers
    Query("q11", "Can I get a static IP address for my home server?",
          "NONE", []),

    # Harder: query that could match multiple articles
    Query("q12", "My bill is too high, I want to either downgrade or cancel.",
          "KB003", ["Basic", "$49", "cancel"]),
]


# ─────────────────────────────────────────────────────────────────────────────
# MOCK LLM (do NOT modify)
# ─────────────────────────────────────────────────────────────────────────────

def mock_llm_call(prompt: str) -> str:
    """
    Simulates an LLM generating an answer from retrieved context.
    Returns a sensible response based on which articles appear in the prompt.
    """
    if "KB001" in prompt and "reset" in prompt.lower():
        return json.dumps({
            "answer": "To reset your modem, locate the small reset button on the back and press it with a paperclip for 10 seconds. Wait 60 seconds for the modem to reboot — all lights should return to solid green. Then test your connection.",
            "source_article": "KB001",
            "confidence": "high",
        })
    elif "KB003" in prompt and ("plan" in prompt.lower() or "basic" in prompt.lower()):
        return json.dumps({
            "answer": "The Basic plan is $49/month with 100 Mbps and a 1 TB data cap. The Plus plan is $69/month with 300 Mbps, unlimited data, and a WiFi 6 router. The Premium plan is $99/month with 1 Gbps, unlimited data, and a mesh WiFi system. All plans are month-to-month with no contracts.",
            "source_article": "KB003",
            "confidence": "high",
        })
    elif "KB002" in prompt and ("payment" in prompt.lower() or "bill" in prompt.lower() or "due" in prompt.lower()):
        return json.dumps({
            "answer": "Payment is due by the 15th of each month. If payment is late by more than 5 days, a $9.99 late fee applies. After 30 days past due, service may be suspended. You can request a payment extension of up to 10 days once per quarter.",
            "source_article": "KB002",
            "confidence": "high",
        })
    elif "KB005" in prompt and ("travel" in prompt.lower() or "international" in prompt.lower() or "roam" in prompt.lower()):
        return json.dumps({
            "answer": "For traveling to Europe, you can purchase a travel pass. The Day Pass is $10/day, the Week Pass is $40/week, or the Month Pass is $80/month — all include unlimited calls, texts, and data in 30+ countries. Text TRAVEL to 611 before departure to activate. Without a pass, roaming charges are $2.99/minute.",
            "source_article": "KB005",
            "confidence": "high",
        })
    elif "KB006" in prompt and ("return" in prompt.lower() or "equipment" in prompt.lower()):
        return json.dumps({
            "answer": "You have 14 days after cancellation to return leased equipment. You can drop it off at any Meridian retail store or request a prepaid shipping label through your online account. Equipment must include all original cables. Late returns (14-30 days) incur a $25 fee per device.",
            "source_article": "KB006",
            "confidence": "high",
        })
    elif "KB007" in prompt and ("slow" in prompt.lower() or "wifi" in prompt.lower() or "speed" in prompt.lower()):
        return json.dumps({
            "answer": "Since your ethernet speed is fine, the issue is WiFi-related. Move your router to a central location away from walls and electronics. Check if your device supports 5 GHz (faster, shorter range) vs 2.4 GHz (slower, longer range). Also check how many devices are connected — too many will slow things down.",
            "source_article": "KB007",
            "confidence": "high",
        })
    elif "KB010" in prompt and "mov" in prompt.lower():
        return json.dumps({
            "answer": "Yes, you can transfer your service when you move. Call us at least 7 days before your move date. A technician visit is typically scheduled within 3-5 business days. There's no charge if you keep the same plan. If service isn't available at the new address, you can cancel with no fees — it's a no-fault cancellation.",
            "source_article": "KB010",
            "confidence": "high",
        })
    elif "KB008" in prompt and ("autopay" in prompt.lower() or "discount" in prompt.lower()):
        return json.dumps({
            "answer": "You can save $5/month by enrolling in AutoPay, which automatically charges your payment method on the due date. Set it up at my.meridianwireless.com under Billing > Payment Methods, or an agent can enable it by phone.",
            "source_article": "KB008",
            "confidence": "high",
        })
    elif "KB011" in prompt and ("block" in prompt.lower() or "parent" in prompt.lower() or "kid" in prompt.lower()):
        return json.dumps({
            "answer": "You can use Meridian Safe, our free parental controls app. It lets you block website categories, set per-device time limits, and schedule bedtime internet shutoffs. Download the app and log in with your Meridian credentials to get started. It works on all devices connected to your home WiFi.",
            "source_article": "KB011",
            "confidence": "high",
        })
    elif "KB009" in prompt and ("outage" in prompt.lower() or "down" in prompt.lower()):
        return json.dumps({
            "answer": "You can check for outages at status.meridianwireless.com or by texting OUTAGE to 611. If there is an outage lasting more than 4 hours, Premium plan customers get an automatic credit. Basic and Plus customers can request a credit within 7 days.",
            "source_article": "KB009",
            "confidence": "high",
        })
    else:
        return json.dumps({
            "answer": "I don't have enough information in our knowledge base to answer that specific question. Let me check with a specialist and get back to you.",
            "source_article": None,
            "confidence": "low",
        })


# ─────────────────────────────────────────────────────────────────────────────
# PART 1: KNOWLEDGE BASE CHUNKING                                 [10 minutes]
# ─────────────────────────────────────────────────────────────────────────────
#
# Split articles into smaller chunks for retrieval. Why chunk?
# - Smaller chunks = more precise matching (a full article might be about
#   many things; a chunk focuses on one subtopic)
# - Token budget: the retrieved chunks are injected into the LLM prompt,
#   so smaller chunks use fewer tokens
#
# At Cresta, the KB is updated frequently as products change. Chunking
# determines retrieval quality — bad chunks = wrong articles surfaced.
# ─────────────────────────────────────────────────────────────────────────────

def chunk_articles(
    articles: List[Article],
    chunk_size: int = 200,
    overlap: int = 50,
) -> List[Chunk]:
    """
    Split each article into overlapping text chunks.

    Args:
        articles: list of KB articles
        chunk_size: target chunk size in WORDS (not characters)
        overlap: number of words to overlap between consecutive chunks

    Returns:
        List of Chunk objects

    Requirements:
      - Split on word boundaries (not mid-word)
      - Each chunk should have ~chunk_size words (last chunk can be shorter)
      - Consecutive chunks overlap by ~overlap words for context continuity
      - Each Chunk must reference its parent article_id and article_title
      - chunk_id format: "{article_id}_chunk_{position}" (e.g. "KB001_chunk_0")

    HINT: A simple approach is to split text into words, then take
    sliding windows of chunk_size words with stride = chunk_size - overlap.
    """
    # TODO: Implement this
    raise NotImplementedError("Part 1: Implement chunk_articles")


# ─────────────────────────────────────────────────────────────────────────────
# PART 2: RETRIEVAL WITH TF-IDF SIMILARITY                        [20 minutes]
# ─────────────────────────────────────────────────────────────────────────────
#
# Build a simple retrieval engine using TF-IDF vectors and cosine similarity.
# In production, you'd use sentence-transformers + FAISS, but the architecture
# is the same: encode → index → query → rank.
#
# At Cresta, retrieval must happen in <200ms of the 800ms budget (leaving
# 600ms for generation). Speed matters as much as accuracy.
# ─────────────────────────────────────────────────────────────────────────────

def tokenize(text: str) -> List[str]:
    """
    Simple tokenizer: lowercase, split on non-alphanumeric, remove stopwords.
    Returns a list of tokens.
    """
    # TODO: Implement this
    raise NotImplementedError("Part 2a: Implement tokenize")


class TFIDFRetriever:
    """
    A from-scratch TF-IDF retrieval engine.

    In an interview, using sklearn's TfidfVectorizer is fine. But implementing
    it from scratch shows you understand what's happening under the hood —
    which matters when debugging retrieval quality issues at Cresta.
    """

    def __init__(self):
        self.chunks: List[Chunk] = []
        self.vocab: Dict[str, int] = {}     # token -> index
        self.idf: Dict[str, float] = {}     # token -> IDF score
        self.tfidf_vectors: List[Dict[str, float]] = []  # sparse vectors per chunk

    def index(self, chunks: List[Chunk]) -> None:
        """
        Build the TF-IDF index from a list of chunks.

        Steps:
          1. Tokenize each chunk
          2. Compute document frequency (DF) for each token
          3. Compute IDF = log(N / DF) for each token
          4. For each chunk, compute TF-IDF vector (TF * IDF per token)

        Store everything so retrieve() can use it.
        """
        # TODO: Implement this
        raise NotImplementedError("Part 2b: Implement TFIDFRetriever.index")

    def retrieve(self, query: str, top_k: int = 3) -> List[Tuple[Chunk, float]]:
        """
        Find the top_k most similar chunks to the query.

        Steps:
          1. Tokenize the query
          2. Compute query TF-IDF vector (using the same IDF from indexing)
          3. Compute cosine similarity between query and every indexed chunk
          4. Return top_k chunks sorted by descending similarity

        Returns: list of (chunk, similarity_score) tuples

        Cosine similarity = dot(A, B) / (|A| * |B|)
        Since vectors are sparse dicts, only iterate over shared keys.
        """
        # TODO: Implement this
        raise NotImplementedError("Part 2c: Implement TFIDFRetriever.retrieve")


# ─────────────────────────────────────────────────────────────────────────────
# PART 3: GROUNDED ANSWER GENERATION                               [15 minutes]
# ─────────────────────────────────────────────────────────────────────────────
#
# Use retrieved chunks to generate an answer. The answer must be:
#   1. Grounded in the retrieved text (no hallucination)
#   2. Cite which article it came from
#   3. Say "I don't know" if retrieval confidence is low
# ─────────────────────────────────────────────────────────────────────────────

def build_rag_prompt(query: str, retrieved_chunks: List[Tuple[Chunk, float]]) -> str:
    """
    Build the prompt for the LLM, injecting retrieved context.

    Requirements:
      - Include the customer's question
      - Include the retrieved chunks with their source article IDs
      - Instruct the model to ONLY use the provided context
      - Instruct the model to return JSON: {answer, source_article, confidence}
      - If no chunks are relevant, tell the model to say "not found"

    Return the complete prompt string.
    """
    # TODO: Implement this
    raise NotImplementedError("Part 3a: Implement build_rag_prompt")


def generate_answer(
    query: str,
    retriever: TFIDFRetriever,
    top_k: int = 3,
    confidence_threshold: float = 0.1,
) -> Dict:
    """
    End-to-end RAG pipeline: retrieve → build prompt → generate → parse.

    Args:
        query: customer's question
        retriever: indexed TFIDFRetriever
        top_k: number of chunks to retrieve
        confidence_threshold: minimum similarity score to consider relevant

    Returns:
    {
        "answer": str,
        "source_article": str or None,
        "confidence": "high" | "low",
        "retrieved_chunks": [{"chunk_id": str, "score": float}, ...],
    }

    If the top retrieved chunk's score is below confidence_threshold,
    return a "not found" answer without calling the LLM.
    """
    # TODO: Implement this
    raise NotImplementedError("Part 3b: Implement generate_answer")


# ─────────────────────────────────────────────────────────────────────────────
# PART 4: EVALUATION                                               [15 minutes]
# ─────────────────────────────────────────────────────────────────────────────
#
# Measure how well your RAG system works on the test queries.
# ─────────────────────────────────────────────────────────────────────────────

def evaluate_retrieval(
    retriever: TFIDFRetriever,
    queries: List[Query],
    top_k: int = 3,
) -> Dict:
    """
    Evaluate retrieval quality.

    For each query, check if the expected article appears in the top_k
    retrieved chunks.

    Returns:
    {
        "total_queries": int,
        "recall_at_k": float,       # % of queries where correct article is in top_k
        "recall_at_1": float,       # % where correct article is the #1 result
        "mrr": float,               # Mean Reciprocal Rank
        "per_query": [
            {
                "query_id": str,
                "expected": str,
                "retrieved_top3": [str, str, str],  # article IDs
                "hit": bool,  # correct article in top_k?
                "rank": int,  # rank of correct article (0 if not found)
            }
        ]
    }

    MRR (Mean Reciprocal Rank):
      For each query, reciprocal rank = 1/rank if the correct article is found,
      else 0. MRR = average of reciprocal ranks across all queries.
      Example: if correct article is rank 1 → 1/1 = 1.0; rank 3 → 1/3 = 0.33
    """
    # TODO: Implement this
    raise NotImplementedError("Part 4a: Implement evaluate_retrieval")


def evaluate_answers(
    retriever: TFIDFRetriever,
    queries: List[Query],
) -> Dict:
    """
    Evaluate end-to-end answer quality.

    For each query:
      - Check if the answer contains expected keywords
      - Check if "not found" queries correctly return low confidence
      - Check source attribution accuracy

    Returns:
    {
        "total_queries": int,
        "keyword_recall": float,     # avg % of expected keywords found in answer
        "source_accuracy": float,    # % of answers citing the correct article
        "not_found_accuracy": float, # % of unanswerable queries correctly declined
        "per_query": [
            {
                "query_id": str,
                "answer_preview": str,
                "keyword_hits": int,
                "keyword_total": int,
                "correct_source": bool,
            }
        ]
    }
    """
    # TODO: Implement this
    raise NotImplementedError("Part 4b: Implement evaluate_answers")


# ─────────────────────────────────────────────────────────────────────────────
# RUNNER
# ─────────────────────────────────────────────────────────────────────────────

def main():
    print("=" * 60)
    print("CRESTA MLE — RAG Knowledge Retrieval")
    print("=" * 60)

    # ── Part 1 ──────────────────────────────────────────────────────
    print("\n📚 PART 1: Chunking")
    print("-" * 40)
    chunks = []
    try:
        chunks = chunk_articles(KB_ARTICLES, chunk_size=100, overlap=20)
        print(f"  Created {len(chunks)} chunks from {len(KB_ARTICLES)} articles")
        # Show a sample
        for c in chunks[:3]:
            preview = c.text[:80] + "..." if len(c.text) > 80 else c.text
            print(f"    {c.chunk_id}: \"{preview}\"")
        print("  ✅ Part 1 passed")
    except NotImplementedError:
        print("  ⬜ Part 1: Not implemented yet")

    # ── Part 2 ──────────────────────────────────────────────────────
    print(f"\n🔍 PART 2: Retrieval")
    print("-" * 40)
    retriever = None
    try:
        if not chunks:
            raise NotImplementedError("Need Part 1 first")
        retriever = TFIDFRetriever()
        retriever.index(chunks)
        print(f"  Indexed {len(chunks)} chunks, vocab size: {len(retriever.vocab)}")

        # Test a query
        results = retriever.retrieve("How do I reset my modem?", top_k=3)
        print(f"  Test query: 'How do I reset my modem?'")
        for chunk, score in results:
            print(f"    {chunk.chunk_id} ({chunk.article_title}): {score:.3f}")
        print("  ✅ Part 2 passed")
    except NotImplementedError:
        print("  ⬜ Part 2: Not implemented yet")
    except Exception as e:
        print(f"  ❌ Part 2 error: {e}")

    # ── Part 3 ──────────────────────────────────────────────────────
    print(f"\n💬 PART 3: Answer Generation")
    print("-" * 40)
    try:
        if retriever is None:
            raise NotImplementedError("Need Part 2 first")
        for q in TEST_QUERIES[:3]:
            result = generate_answer(q.text, retriever)
            src = result.get("source_article", "none")
            conf = result.get("confidence", "?")
            ans = result.get("answer", "")[:80]
            print(f"  {q.id}: [{conf}] source={src}")
            print(f"    → \"{ans}...\"")

        # Test the "not found" query
        nf = generate_answer(TEST_QUERIES[10].text, retriever)  # q11: static IP
        print(f"  q11 (unanswerable): confidence={nf.get('confidence')}")
        print("  ✅ Part 3 passed")
    except NotImplementedError:
        print("  ⬜ Part 3: Not implemented yet")
    except Exception as e:
        print(f"  ❌ Part 3 error: {e}")

    # ── Part 4 ──────────────────────────────────────────────────────
    print(f"\n📋 PART 4: Evaluation")
    print("-" * 40)
    try:
        if retriever is None:
            raise NotImplementedError("Need Part 2 first")
        ret_eval = evaluate_retrieval(retriever, TEST_QUERIES, top_k=3)
        print(f"  Retrieval:")
        print(f"    Recall@3: {ret_eval['recall_at_k']:.1%}")
        print(f"    Recall@1: {ret_eval['recall_at_1']:.1%}")
        print(f"    MRR:      {ret_eval['mrr']:.3f}")

        # Show misses
        for pq in ret_eval["per_query"]:
            if not pq["hit"] and pq["expected"] != "NONE":
                print(f"    ❌ {pq['query_id']}: expected {pq['expected']}, got {pq['retrieved_top3']}")

        print("  ✅ Part 4a passed")
    except NotImplementedError:
        print("  ⬜ Part 4a: Not implemented yet")
    except Exception as e:
        print(f"  ❌ Part 4a error: {e}")

    try:
        if retriever is None:
            raise NotImplementedError("Need Part 2 first")
        ans_eval = evaluate_answers(retriever, TEST_QUERIES)
        print(f"\n  Answer Quality:")
        print(f"    Keyword recall:     {ans_eval['keyword_recall']:.1%}")
        print(f"    Source accuracy:     {ans_eval['source_accuracy']:.1%}")
        print(f"    Not-found accuracy: {ans_eval['not_found_accuracy']:.1%}")
        print("  ✅ Part 4b passed")
    except NotImplementedError:
        print("  ⬜ Part 4b: Not implemented yet")
    except Exception as e:
        print(f"  ❌ Part 4b error: {e}")

    print("\n" + "=" * 60)


if __name__ == "__main__":
    main()

In [ ]:
"""
================================================================================
SIMPLE SOLUTION — Cresta RAG Knowledge Retrieval (realistic for 1 hour)
================================================================================
"""

import json
import re
import math
from typing import Dict, List, Tuple, Optional
from collections import Counter

from cresta_rag_problem import (
    Article, Chunk, Query,
    KB_ARTICLES, TEST_QUERIES,
    TFIDFRetriever,
    mock_llm_call,
)


# ═══════════════════════════════════════════════════════════════════════
# PART 1 — chunking (sliding window over words)
# ═══════════════════════════════════════════════════════════════════════

def chunk_articles(articles, chunk_size=200, overlap=50):
    chunks = []

    for article in articles:
        words = article.content.split()

        # If the article is shorter than chunk_size, it's one chunk
        if len(words) <= chunk_size:
            chunks.append(Chunk(
                chunk_id=f"{article.id}_chunk_0",
                article_id=article.id,
                article_title=article.title,
                text=article.content,
                position=0,
            ))
            continue

        # Sliding window
        stride = chunk_size - overlap
        pos = 0
        start = 0
        while start < len(words):
            end = min(start + chunk_size, len(words))
            chunk_text = " ".join(words[start:end])
            chunks.append(Chunk(
                chunk_id=f"{article.id}_chunk_{pos}",
                article_id=article.id,
                article_title=article.title,
                text=chunk_text,
                position=pos,
            ))
            pos += 1
            start += stride
            if end == len(words):
                break

    return chunks


# ═══════════════════════════════════════════════════════════════════════
# PART 2 — TF-IDF retriever from scratch
# ═══════════════════════════════════════════════════════════════════════

STOPWORDS = {
    "the", "a", "an", "is", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "do", "does", "did", "will", "would", "could",
    "should", "may", "might", "can", "shall", "to", "of", "in", "for",
    "on", "with", "at", "by", "from", "as", "into", "through", "during",
    "before", "after", "and", "but", "or", "nor", "not", "so", "yet",
    "both", "either", "neither", "each", "every", "all", "any", "few",
    "more", "most", "other", "some", "such", "no", "only", "own", "same",
    "than", "too", "very", "just", "because", "if", "when", "where",
    "how", "what", "which", "who", "whom", "this", "that", "these",
    "those", "it", "its", "i", "me", "my", "we", "our", "you", "your",
    "he", "him", "his", "she", "her", "they", "them", "their", "about",
}


def tokenize(text):
    """Lowercase, keep only alphanumeric, remove stopwords."""
    words = re.findall(r'[a-z0-9]+', text.lower())
    return [w for w in words if w not in STOPWORDS and len(w) > 1]


class SimpleTFIDFRetriever(TFIDFRetriever):
    """Inherit from the stub class so the runner works."""

    def __init__(self):
        self.chunks = []
        self.vocab = {}
        self.idf = {}
        self.tfidf_vectors = []
        self._token_lists = []  # store tokenized chunks

    def index(self, chunks):
        self.chunks = chunks
        N = len(chunks)

        # Tokenize all chunks
        self._token_lists = [tokenize(c.text) for c in chunks]

        # Document frequency: how many chunks contain each token
        df = Counter()
        for tokens in self._token_lists:
            unique_tokens = set(tokens)
            for t in unique_tokens:
                df[t] += 1

        # IDF = log(N / df)   (add 1 to avoid division by zero on query-only terms)
        self.idf = {t: math.log(N / count) for t, count in df.items()}

        # Build vocab
        self.vocab = {t: i for i, t in enumerate(self.idf.keys())}

        # TF-IDF vector for each chunk (sparse dict: token -> score)
        self.tfidf_vectors = []
        for tokens in self._token_lists:
            tf = Counter(tokens)
            total = len(tokens) if tokens else 1
            vec = {}
            for t, count in tf.items():
                if t in self.idf:
                    vec[t] = (count / total) * self.idf[t]
            self.tfidf_vectors.append(vec)

    def retrieve(self, query, top_k=3):
        # Build query vector
        tokens = tokenize(query)
        tf = Counter(tokens)
        total = len(tokens) if tokens else 1
        query_vec = {}
        for t, count in tf.items():
            if t in self.idf:
                query_vec[t] = (count / total) * self.idf[t]

        if not query_vec:
            return []

        # Cosine similarity with every chunk
        scores = []
        query_norm = math.sqrt(sum(v ** 2 for v in query_vec.values()))

        for i, chunk_vec in enumerate(self.tfidf_vectors):
            # Dot product (only shared keys)
            dot = sum(query_vec[t] * chunk_vec.get(t, 0) for t in query_vec)
            chunk_norm = math.sqrt(sum(v ** 2 for v in chunk_vec.values()))

            if query_norm > 0 and chunk_norm > 0:
                sim = dot / (query_norm * chunk_norm)
            else:
                sim = 0.0

            scores.append((self.chunks[i], sim))

        # Sort by similarity descending
        scores.sort(key=lambda x: -x[1])
        return scores[:top_k]


# ═══════════════════════════════════════════════════════════════════════
# PART 3 — grounded answer generation
# ═══════════════════════════════════════════════════════════════════════

def build_rag_prompt(query, retrieved_chunks):
    context_parts = []
    for chunk, score in retrieved_chunks:
        context_parts.append(f"[Source: {chunk.article_id} — {chunk.article_title}]\n{chunk.text}")

    context = "\n\n".join(context_parts)

    return f"""Answer the customer's question using ONLY the context below.
If the context doesn't contain the answer, say you don't have that information.
Return JSON: {{"answer": "...", "source_article": "KBXXX", "confidence": "high/low"}}

Context:
{context}

Customer question: {query}

JSON response:"""


def generate_answer(query, retriever, top_k=3, confidence_threshold=0.1):
    # Step 1: Retrieve
    results = retriever.retrieve(query, top_k=top_k)

    # Step 2: Check if top result meets threshold
    if not results or results[0][1] < confidence_threshold:
        return {
            "answer": "I don't have information about that in our knowledge base.",
            "source_article": None,
            "confidence": "low",
            "retrieved_chunks": [],
        }

    # Step 3: Build prompt and call LLM
    prompt = build_rag_prompt(query, results)
    raw = mock_llm_call(prompt)

    # Step 4: Parse response
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        parsed = {"answer": raw, "source_article": None, "confidence": "low"}

    parsed["retrieved_chunks"] = [
        {"chunk_id": c.chunk_id, "score": round(s, 3)} for c, s in results
    ]

    return parsed


# ═══════════════════════════════════════════════════════════════════════
# PART 4 — evaluation
# ═══════════════════════════════════════════════════════════════════════

def evaluate_retrieval(retriever, queries, top_k=3):
    per_query = []
    hits_at_k = 0
    hits_at_1 = 0
    reciprocal_ranks = []

    for q in queries:
        results = retriever.retrieve(q.text, top_k=top_k)
        retrieved_ids = [c.article_id for c, _ in results]

        if q.expected_article == "NONE":
            # For unanswerable queries, skip retrieval metrics
            per_query.append({
                "query_id": q.id,
                "expected": "NONE",
                "retrieved_top3": retrieved_ids[:3],
                "hit": True,  # not applicable
                "rank": 0,
            })
            reciprocal_ranks.append(0)
            continue

        # Find rank of expected article
        rank = 0
        for i, aid in enumerate(retrieved_ids):
            if aid == q.expected_article:
                rank = i + 1
                break

        hit = rank > 0 and rank <= top_k
        if hit:
            hits_at_k += 1
        if rank == 1:
            hits_at_1 += 1
        reciprocal_ranks.append(1.0 / rank if rank > 0 else 0.0)

        per_query.append({
            "query_id": q.id,
            "expected": q.expected_article,
            "retrieved_top3": retrieved_ids[:3],
            "hit": hit,
            "rank": rank,
        })

    answerable = sum(1 for q in queries if q.expected_article != "NONE")

    return {
        "total_queries": len(queries),
        "recall_at_k": hits_at_k / max(answerable, 1),
        "recall_at_1": hits_at_1 / max(answerable, 1),
        "mrr": sum(reciprocal_ranks) / max(len(reciprocal_ranks), 1),
        "per_query": per_query,
    }


def evaluate_answers(retriever, queries):
    per_query = []
    keyword_scores = []
    source_correct = 0
    not_found_correct = 0
    not_found_total = 0
    answerable_total = 0

    for q in queries:
        result = generate_answer(q.text, retriever)
        answer_lower = result.get("answer", "").lower()

        if q.expected_article == "NONE":
            # Unanswerable — should return low confidence
            not_found_total += 1
            if result.get("confidence") == "low" or result.get("source_article") is None:
                not_found_correct += 1
            continue

        answerable_total += 1

        # Keyword recall
        hits = sum(1 for kw in q.expected_answer_keywords if kw.lower() in answer_lower)
        total_kw = len(q.expected_answer_keywords)
        keyword_scores.append(hits / max(total_kw, 1))

        # Source accuracy
        correct_src = result.get("source_article") == q.expected_article
        if correct_src:
            source_correct += 1

        per_query.append({
            "query_id": q.id,
            "answer_preview": result.get("answer", "")[:80],
            "keyword_hits": hits,
            "keyword_total": total_kw,
            "correct_source": correct_src,
        })

    return {
        "total_queries": len(queries),
        "keyword_recall": sum(keyword_scores) / max(len(keyword_scores), 1),
        "source_accuracy": source_correct / max(answerable_total, 1),
        "not_found_accuracy": not_found_correct / max(not_found_total, 1),
        "per_query": per_query,
    }


# ═══════════════════════════════════════════════════════════════════════
# RUNNER
# ═══════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    print("=" * 60)
    print("SIMPLE SOLUTION — RAG Knowledge Retrieval")
    print("=" * 60)

    # Part 1
    chunks = chunk_articles(KB_ARTICLES, chunk_size=100, overlap=20)
    print(f"\n📚 Part 1: {len(chunks)} chunks from {len(KB_ARTICLES)} articles")

    # Part 2
    retriever = SimpleTFIDFRetriever()
    retriever.index(chunks)
    print(f"🔍 Part 2: Indexed, vocab={len(retriever.vocab)} tokens")

    # Quick test
    results = retriever.retrieve("How do I reset my modem?", top_k=3)
    print(f"   Test: 'How do I reset my modem?'")
    for chunk, score in results:
        print(f"     {chunk.article_id} ({chunk.article_title[:30]}): {score:.3f}")

    # Part 3
    print(f"\n💬 Part 3: Generating answers")
    for q in TEST_QUERIES[:4]:
        result = generate_answer(q.text, retriever)
        print(f"   {q.id}: [{result['confidence']}] {result['answer'][:70]}...")

    # Part 4
    print(f"\n📋 Part 4: Evaluation")
    ret_eval = evaluate_retrieval(retriever, TEST_QUERIES, top_k=3)
    print(f"   Recall@3: {ret_eval['recall_at_k']:.1%}")
    print(f"   Recall@1: {ret_eval['recall_at_1']:.1%}")
    print(f"   MRR:      {ret_eval['mrr']:.3f}")

    for pq in ret_eval["per_query"]:
        if not pq["hit"] and pq["expected"] != "NONE":
            print(f"   ❌ {pq['query_id']}: expected {pq['expected']}, got {pq['retrieved_top3']}")

    ans_eval = evaluate_answers(retriever, TEST_QUERIES)
    print(f"\n   Keyword recall:     {ans_eval['keyword_recall']:.1%}")
    print(f"   Source accuracy:    {ans_eval['source_accuracy']:.1%}")
    print(f"   Not-found accuracy: {ans_eval['not_found_accuracy']:.1%}")

    print("\n" + "=" * 60)